# Feature Engineering

Prepare the customer support ticket dataset for machine learning by selecting, cleaning, and transforming features.

In [4]:
import pandas as pd

from src.data_loader import load_csv
from src.data_validator import validate_dataframe

In [5]:
df = load_csv("../data/raw/customer_support_tickets.csv")

validate_dataframe(df)

df.head()

2026-07-07 14:06:02,869 | INFO | src.data_loader | Loading dataset: ..\data\raw\customer_support_tickets.csv
2026-07-07 14:06:02,908 | INFO | src.data_loader | Dataset loaded successfully (8469 rows, 17 columns)
2026-07-07 14:06:02,908 | INFO | src.data_validator | Starting dataset validation.
2026-07-07 14:06:02,910 | INFO | src.data_validator | All required columns are present.
2026-07-07 14:06:02,916 | INFO | src.data_validator | No missing values found.
2026-07-07 14:06:02,917 | INFO | src.data_validator | No duplicate Ticket IDs found.
2026-07-07 14:06:02,920 | INFO | src.data_validator | Priority values are valid.
2026-07-07 14:06:02,921 | INFO | src.data_validator | Ticket types are valid.
2026-07-07 14:06:02,922 | INFO | src.data_validator | Ticket status values are valid.
2026-07-07 14:06:02,923 | INFO | src.data_validator | Dataset validation completed successfully.


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


## Dataset Shape

In [6]:
df.shape

(8469, 17)

In [7]:
selected_columns = [
    "Ticket Subject",
    "Ticket Description",
    "Ticket Type",
    "Ticket Priority",
    "Ticket Channel",
    "Customer Age",
    "Customer Gender",
    "Product Purchased",
    "Customer Satisfaction Rating",
]

In [8]:
df = df[selected_columns]

df.head()

,Ticket Subject,Ticket Description,Ticket Type,Ticket Priority,Ticket Channel,Customer Age,Customer Gender,Product Purchased,Customer Satisfaction Rating
0,Product setup,I'm having an issue with the {product_purchase...,Technical issue,Critical,Social media,32,Other,GoPro Hero,NaN
1,Peripheral compatibility,I'm having an issue with the {product_purchase...,Technical issue,Critical,Chat,42,Female,LG Smart TV,NaN
2,Network problem,I'm facing a problem with my {product_purchase...,Technical issue,Low,Social media,48,Other,Dell XPS,3.0
3,Account access,I'm having an issue with the {product_purchase...,Billing inquiry,Low,Social media,27,Female,Microsoft Office,3.0
4,Data loss,I'm having an issue with the {product_purchase...,Billing inquiry,Low,Email,67,Female,Autodesk AutoCAD,1.0


## Missing Value Check

Verify that the selected features do not contain missing values before feature engineering.

In [9]:
df.isnull().sum()

Ticket Subject                     0
Ticket Description                 0
Ticket Type                        0
Ticket Priority                    0
Ticket Channel                     0
Customer Age                       0
Customer Gender                    0
Product Purchased                  0
Customer Satisfaction Rating    5700
dtype: int64

## Remove Missing Target Values

Remove records that do not contain customer satisfaction ratings since they cannot be used for supervised machine learning.

In [10]:
df = df.dropna(subset=["Customer Satisfaction Rating"])

df.shape

(2769, 9)

In [11]:
df.isnull().sum()

Ticket Subject                  0
Ticket Description              0
Ticket Type                     0
Ticket Priority                 0
Ticket Channel                  0
Customer Age                    0
Customer Gender                 0
Product Purchased               0
Customer Satisfaction Rating    0
dtype: int64

## Encode Categorical Features

Convert categorical variables into numerical values so they can be used by machine learning algorithms.

In [12]:
categorical_columns = [
    "Ticket Type",
    "Ticket Priority",
    "Ticket Channel",
    "Customer Gender",
    "Product Purchased"
]

In [13]:
from sklearn.preprocessing import LabelEncoder

In [14]:
label_encoders = {}

for column in categorical_columns:
    encoder = LabelEncoder()
    df[column] = encoder.fit_transform(df[column])

    label_encoders[column] = encoder

In [15]:
df.head()

,Ticket Subject,Ticket Description,Ticket Type,Ticket Priority,Ticket Channel,Customer Age,Customer Gender,Product Purchased,Customer Satisfaction Rating
2,Network problem,I'm facing a problem with my {product_purchase...,4,2,3,48,2,10,3.0
3,Account access,I'm having an issue with the {product_purchase...,0,2,3,27,0,25,3.0
4,Data loss,I'm having an issue with the {product_purchase...,0,2,1,67,0,5,1.0
10,Data loss,I'm having an issue with the {product_purchase...,1,1,2,48,1,30,1.0
11,Software bug,I'm having an issue with the {product_purchase...,2,1,0,51,1,27,1.0


## Verify Data Types

Ensure all features are in the correct numerical format after encoding.

In [16]:
df.dtypes

Ticket Subject                   object
Ticket Description               object
Ticket Type                       int64
Ticket Priority                   int64
Ticket Channel                    int64
Customer Age                      int64
Customer Gender                   int64
Product Purchased                 int64
Customer Satisfaction Rating    float64
dtype: object

## Save Label Encoders

Store the fitted label encoders for future use during inference.

In [17]:
import joblib
import os

In [18]:
os.makedirs("../artifacts", exist_ok=True)

In [19]:
joblib.dump(label_encoders, "../artifacts/label_encoders.pkl")

['../models/label_encoders.pkl']

In [20]:
import os

os.listdir("../artifacts")

['label_encoders.pkl', 'README.md', 'tfidf_vectorizer.pkl']

## Text Feature Engineering

Convert ticket subject and ticket description into numerical features using TF-IDF vectorization.

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [22]:
df["Ticket Text"] = (
    df["Ticket Subject"] + " " + df["Ticket Description"]
)

In [23]:
df[["Ticket Subject", "Ticket Description", "Ticket Text"]].head()

,Ticket Subject,Ticket Description,Ticket Text
2,Network problem,I'm facing a problem with my {product_purchase...,Network problem I'm facing a problem with my {...
3,Account access,I'm having an issue with the {product_purchase...,Account access I'm having an issue with the {p...
4,Data loss,I'm having an issue with the {product_purchase...,Data loss I'm having an issue with the {produc...
10,Data loss,I'm having an issue with the {product_purchase...,Data loss I'm having an issue with the {produc...
11,Software bug,I'm having an issue with the {product_purchase...,Software bug I'm having an issue with the {pro...


In [24]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=500
)

In [25]:
X_text = tfidf.fit_transform(df["Ticket Text"])

In [26]:
X_text.shape

(2769, 500)

In [27]:
joblib.dump(tfidf, "../artifacts/tfidf_vectorizer.pkl")

['../models/tfidf_vectorizer.pkl']

In [28]:
os.listdir("../artifacts")

['label_encoders.pkl', 'README.md', 'tfidf_vectorizer.pkl']

# Train-Test Split

Split the dataset into training and testing sets to evaluate the machine learning model on unseen data.

In [29]:
X = df.drop("Customer Satisfaction Rating", axis=1)
y = df["Customer Satisfaction Rating"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (2769, 9)
Target Shape: (2769,)


# Create Training and Testing Sets

Use an 80-20 split to train the model while keeping a separate testing dataset for evaluation.

In [30]:
from sklearn.model_selection import train_test_split

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [32]:
print("Training Features:", X_train.shape)
print("Testing Features :", X_test.shape)

print("Training Labels :", y_train.shape)
print("Testing Labels  :", y_test.shape)

Training Features: (2215, 9)
Testing Features : (554, 9)
Training Labels : (2215,)
Testing Labels  : (554,)


# Save Processed Dataset

Save the processed training and testing datasets so they can be used during model development without repeating the feature engineering process.

In [33]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [34]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

In [35]:
os.listdir("../data/processed")

['README.md', 'X_test.csv', 'X_train.csv', 'y_test.csv', 'y_train.csv']

## Load Feature Engineering Objects

Load the saved TF-IDF vectorizer and label encoders produced during the feature engineering stage.

In [36]:
import joblib

tfidf = joblib.load("../artifacts/tfidf_vectorizer.pkl")
label_encoders = joblib.load("../artifacts/label_encoders.pkl")

## Transform Text Features

Convert ticket descriptions into TF-IDF numerical vectors.

In [37]:
X_text = tfidf.transform(df["Ticket Description"])

In [38]:
numeric_features = df.drop(
    columns=[
        "Ticket Description",
        "Ticket Subject",
        "Customer Satisfaction Rating"
    ]
)

numeric_features.head()

,Ticket Type,Ticket Priority,Ticket Channel,Customer Age,Customer Gender,Product Purchased,Ticket Text
2,4,2,3,48,2,10,Network problem I'm facing a problem with my {...
3,0,2,3,27,0,25,Account access I'm having an issue with the {p...
4,0,2,1,67,0,5,Data loss I'm having an issue with the {produc...
10,1,1,2,48,1,30,Data loss I'm having an issue with the {produc...
11,2,1,0,51,1,27,Software bug I'm having an issue with the {pro...


In [39]:
numeric_features.dtypes

Ticket Type           int64
Ticket Priority       int64
Ticket Channel        int64
Customer Age          int64
Customer Gender       int64
Product Purchased     int64
Ticket Text          object
dtype: object

In [40]:
numeric_features = df.drop(
    columns=[
        "Ticket Description",
        "Ticket Subject",
        "Ticket Text",
        "Customer Satisfaction Rating"
    ],
    errors="ignore"
)

numeric_features.head()

,Ticket Type,Ticket Priority,Ticket Channel,Customer Age,Customer Gender,Product Purchased
2,4,2,3,48,2,10
3,0,2,3,27,0,25
4,0,2,1,67,0,5
10,1,1,2,48,1,30
11,2,1,0,51,1,27


In [41]:
numeric_features.dtypes

Ticket Type          int64
Ticket Priority      int64
Ticket Channel       int64
Customer Age         int64
Customer Gender      int64
Product Purchased    int64
dtype: object

In [42]:
from scipy.sparse import hstack

X = hstack([X_text, numeric_features])

y = df["Customer Satisfaction Rating"]

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [44]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [45]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [46]:
y_pred = linear_model.predict(X_test)

In [47]:
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = mse ** 0.5

r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 1.3165020833352263
MSE : 2.493378090676014
RMSE: 1.5790434100036688
R²  : -0.2640508193377933


## Random Forest Regression Model
Train a Random Forest model and compare its performance with Linear Regression.

In [48]:
from sklearn.ensemble import RandomForestRegressor

In [49]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train.toarray(), y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [50]:
rf_pred = rf_model.predict(X_test.toarray())

In [51]:
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = rf_mse ** 0.5
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Results")
print("---------------------")
print("MAE :", rf_mae)
print("MSE :", rf_mse)
print("RMSE:", rf_rmse)
print("R²  :", rf_r2)

Random Forest Results
---------------------
MAE : 1.243158844765343
MSE : 2.125512454873646
RMSE: 1.4579137336871637
R²  : -0.07755649660307773


## Train Decision Tree Regressor

A Decision Tree Regressor learns nonlinear relationships by recursively splitting the feature space. Unlike Linear Regression, it does not assume a linear relationship between the input features and the target variable.

In [52]:
from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(
    random_state=42
)

dt_model.fit(
    X_train.toarray(),
    y_train
)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


### Generate Predictions

Use the trained Decision Tree model to predict customer satisfaction ratings on the test dataset.

In [53]:
dt_pred = dt_model.predict(
    X_test.toarray()
)

### Evaluate Decision Tree Performance

In [54]:
dt_mae = mean_absolute_error(
    y_test,
    dt_pred
)

dt_mse = mean_squared_error(
    y_test,
    dt_pred
)

dt_rmse = dt_mse ** 0.5

dt_r2 = r2_score(
    y_test,
    dt_pred
)

print("Decision Tree Results")
print("----------------------")
print("MAE :", dt_mae)
print("MSE :", dt_mse)
print("RMSE:", dt_rmse)
print("R²  :", dt_r2)

Decision Tree Results
----------------------
MAE : 1.5451263537906137
MSE : 3.7978339350180503
RMSE: 1.94880320582096
R²  : -0.9253618675885897


## Train Gradient Boosting Regressor

Gradient Boosting builds an ensemble of weak decision trees sequentially. Each new tree learns from the errors of the previous trees, often resulting in better predictive performance than a single decision tree.

In [55]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    random_state=42
)

gb_model.fit(
    X_train.toarray(),
    y_train
)

,loss,'squared_error'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


### Generate Predictions

Predict customer satisfaction ratings using the trained Gradient Boosting model.

In [56]:
gb_pred = gb_model.predict(
    X_test.toarray()
)

### Evaluate Gradient Boosting Performance

In [57]:
gb_mae = mean_absolute_error(
    y_test,
    gb_pred
)

gb_mse = mean_squared_error(
    y_test,
    gb_pred
)

gb_rmse = gb_mse ** 0.5

gb_r2 = r2_score(
    y_test,
    gb_pred
)

print("Gradient Boosting Results")
print("-------------------------")
print("MAE :", gb_mae)
print("MSE :", gb_mse)
print("RMSE:", gb_rmse)
print("R²  :", gb_r2)

Gradient Boosting Results
-------------------------
MAE : 1.2224779975805447
MSE : 2.0262428188612382
RMSE: 1.42346156212988
R²  : -0.02723044889852444


# Model Comparison

Multiple regression models were trained and evaluated using the following metrics:

- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- R² Score

The objective is to identify the model with the lowest prediction error and highest explanatory power.

In [58]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Decision Tree",
        "Gradient Boosting"
    ],
    "MAE": [
        mae,
        rf_mae,
        dt_mae,
        gb_mae
    ],
    "MSE": [
        mse,
        rf_mse,
        dt_mse,
        gb_mse
    ],
    "RMSE": [
        rmse,
        rf_rmse,
        dt_rmse,
        gb_rmse
    ],
    "R2 Score": [
        r2,
        rf_r2,
        dt_r2,
        gb_r2
    ]
})

results

,Model,MAE,MSE,RMSE,R2 Score
0,Linear Regression,1.316502,2.493378,1.579043,-0.264051
1,Random Forest,1.243159,2.125512,1.457914,-0.077556
2,Decision Tree,1.545126,3.797834,1.948803,-0.925362
3,Gradient Boosting,1.222478,2.026243,1.423462,-0.027230


In [59]:
results.sort_values(
    by="R2 Score",
    ascending=False
)

,Model,MAE,MSE,RMSE,R2 Score
3,Gradient Boosting,1.222478,2.026243,1.423462,-0.027230
1,Random Forest,1.243159,2.125512,1.457914,-0.077556
0,Linear Regression,1.316502,2.493378,1.579043,-0.264051
2,Decision Tree,1.545126,3.797834,1.948803,-0.925362


## Best Performing Model

Gradient Boosting Regressor achieved the highest R² score among all evaluated models.

Although the R² score remains slightly negative, it performed better than Linear Regression, Decision Tree, and Random Forest on this dataset.

The relatively poor scores indicate that additional feature engineering, larger datasets, or hyperparameter tuning would be required for production-level performance.

In [60]:
from pathlib import Path

Path("../processed").mkdir(exist_ok=True)

In [61]:
import joblib

joblib.dump(
    {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    },
    "../processed/train_test_data.pkl"
)

['../processed/train_test_data.pkl']

In [62]:
from pathlib import Path

Path("../processed/train_test_data.pkl").exists()

True

In [63]:
print(type(X_train))
print(X_train.shape)

if hasattr(X_train, "dtypes"):
    print("\nData types:")
    print(X_train.dtypes.value_counts())

    print("\nObject columns:")
    print(X_train.select_dtypes(include="object").columns.tolist())
else:
    print("X_train is not a pandas DataFrame.")

<class 'scipy.sparse._csr.csr_matrix'>
(2215, 506)
X_train is not a pandas DataFrame.


In [64]:
print(type(X))
print(X.shape)

if hasattr(X, "dtypes"):
    print("\nData types:")
    print(X.dtypes.value_counts())

    print("\nObject columns:")
    print(X.select_dtypes(include="object").columns.tolist())

<class 'scipy.sparse._coo.coo_matrix'>
(2769, 506)


In [65]:
%whos

Variable                    Type                         Data/Info
------------------------------------------------------------------
DecisionTreeRegressor       ABCMeta                      <class 'sklearn.tree._cla<...>s.DecisionTreeRegressor'>
GradientBoostingRegressor   ABCMeta                      <class 'sklearn.ensemble.<...>adientBoostingRegressor'>
LabelEncoder                type                         <class 'sklearn.preproces<...>ing._label.LabelEncoder'>
LinearRegression            ABCMeta                      <class 'sklearn.linear_mo<...>._base.LinearRegression'>
Path                        type                         <class 'pathlib.Path'>
RandomForestRegressor       ABCMeta                      <class 'sklearn.ensemble.<...>t.RandomForestRegressor'>
TfidfVectorizer             type                         <class 'sklearn.feature_e<...>on.text.TfidfVectorizer'>
X                           coo_matrix                   <COOrdinate sparse matrix<...>)	1.0\n  (2768, 505)	

# Feature Engineering Summary

Completed Tasks:

- Missing value validation
- Duplicate removal
- Label encoding
- TF-IDF vectorization
- Feature combination
- Train-test split
- Baseline model evaluation
- Multiple regression model comparison
- Engineered dataset serialization

The processed dataset is now ready for the dedicated Model Training notebook.